# Prática 01/12 · Tribunal do Pipeline: auditar e consertar um experimento
**Aprendizagem Computacional · 4º TIA · Aula 06 · versão INICIAL**

Um colega entregou o experimento abaixo com uma conclusão animadora: *"meu k-NN acerta mais de 80% dos municípios"*.
O código roda sem erro. O problema é que o resultado **não se sustenta**. Hoje vocês são a banca: encontrar os erros, consertar e medir quanto o resultado "encolhe".
É o ensaio para a banca da PPI: as perguntas que vocês vão responder aqui são as mesmas que a banca vai fazer sobre o projeto de vocês.

| Parte | O que você faz | Apostila |
|---|---|---|
| 1 | Carregar (pronto) | — |
| 2 | Rodar o experimento do colega (pronto) e **listar os erros** | cap. 5, 9, 12 |
| 3 | **TODO 1:** um X honesto | cap. 2, 5 |
| 4 | **TODO 2:** separar primeiro, estratificado | cap. 9 |
| 5 | **TODO 3:** escala dentro do Pipeline | cap. 5 |
| 6 | **TODO 4:** escolher o k com validação cruzada no treino | cap. 9, 13 |
| 7 | **TODO 5:** baseline e métricas certas, teste UMA vez | cap. 9, 12 |
| 8 | Comparar: antes × depois (pronto) | — |
| 9 | Verificação final | — |
| 10 | Auditoria do projeto do seu grupo | cap. 24 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42
LIMIAR_ALTA = 25
FEATURES = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO']

## 1 · Carregar os dados (pronto)
Os 142 municípios de MT (TSE, 2026), a mesma base das práticas anteriores.

In [ ]:
ARQUIVO = 'municipios_mt_votos.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-12-01_Aula06_Tribunal_do_Pipeline/dados/municipios_mt_votos.csv'

candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
if local is not None:
    municipios = pd.read_csv(local)
    print('Lido do arquivo:', local.resolve())
else:
    municipios = pd.read_csv(URL)
    print('Lido do GitHub da disciplina')
print(municipios.shape)

## 2 · O experimento do colega (pronto: rode e leia com atenção)
Não altere esta célula. Ela é o "réu".

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# ===== experimento do colega =====
dados = municipios.copy()
dados['PCT_COMPARECIMENTO'] = 100 * dados['QT_COMPARECIMENTO'] / dados['QT_APTOS']   # "uma feature a mais"
y_colega = (dados['ABSTENCAO_PCT'] >= LIMIAR_ALTA).astype(int)
X_colega = dados[['CD_MUNICIPIO', 'PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO',
                  'LOG_ELEITORADO', 'PCT_COMPARECIMENTO']]
X_colega = StandardScaler().fit_transform(X_colega)
Xa, Xb, ya, yb = train_test_split(X_colega, y_colega, test_size=0.25, random_state=SEMENTE)

melhor_k, melhor_acc = None, 0
for k in [1, 3, 5, 7, 9, 11, 15]:
    acc = accuracy_score(yb, KNeighborsClassifier(n_neighbors=k).fit(Xa, ya).predict(Xb))
    if acc > melhor_acc:
        melhor_k, melhor_acc = k, acc

print(f'Conclusão do colega: k-NN com k={melhor_k} acerta {melhor_acc:.0%} dos municípios!')
# ==================================

**Antes de programar, discuta em dupla e escreva numa célula de texto abaixo:** quais são os erros? Há **5 erros graves** (e um sexto, menor).

<details><summary><b>Gabarito da discussão (abra só depois de tentar)</b></summary>

1. **ID como feature:** `CD_MUNICIPIO` é só uma etiqueta. O modelo pode achar padrões falsos nos números do código.
2. **Vazamento do alvo:** `PCT_COMPARECIMENTO` é 100 − abstenção: é a própria resposta com outro nome. E só existe **depois** da eleição, quando já não há o que prever.
3. **Escala antes do split:** o `StandardScaler` aprendeu média e desvio com o teste junto. O teste "vazou" para a preparação.
4. **Hiperparâmetro escolhido no teste:** o `for` testa 7 valores de k **no teste** e fica com o melhor. O teste virou validação, e a nota ficou otimista.
5. **Só acurácia, sem baseline:** 69% de acurácia é o que se ganha chutando "baixa" para todos. Sem baseline e sem recall, não dá para saber se o modelo serve.
6. *(menor)* **Split sem `stratify`:** com 142 linhas, o teste pode ficar com poucos "alta".
</details>

## 3 · TODO 1: um X honesto (cap. 2 e 5)
`X = municipios[FEATURES]` e `y` igual ao do colega. Sem identificador e sem nada que venha do resultado da eleição.

In [ ]:
# TODO 1
y = None
X = None

print(X.shape, '| colunas:', list(X.columns))

## 4 · TODO 2: separar primeiro, estratificado (cap. 9)
`train_test_split(X, y, test_size=0.25, random_state=SEMENTE, stratify=y)`. A partir daqui, o teste fica guardado até a parte 7.

In [ ]:
# TODO 2
X_treino, X_teste, y_treino, y_teste = None, None, None, None

print('treino:', X_treino.shape, '| teste:', X_teste.shape, '| "alta" no teste:', y_teste.sum())

## 5 · TODO 3: escala dentro do Pipeline (cap. 5)
`pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier())`. Dentro do Pipeline, a escala é aprendida só com os dados de treino de cada etapa.

In [ ]:
from sklearn.pipeline import make_pipeline

# TODO 3
pipeline = None

print(pipeline)

## 6 · TODO 4: escolher o k com validação cruzada, só no treino (cap. 9 e 13)
Use `GridSearchCV(pipeline, grade, cv=cv, scoring='f1')` e ajuste em `X_treino, y_treino`. O `cv` já está pronto.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
grade = {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 11, 15]}

# TODO 4
busca = None

print('k escolhido pela CV:', busca.best_params_['kneighborsclassifier__n_neighbors'],
      '| F1 médio na CV:', round(busca.best_score_, 2))

## 7 · TODO 5: baseline e métricas certas, teste UMA vez (cap. 9 e 12)
1. `baseline = DummyClassifier(strategy='most_frequent').fit(X_treino, y_treino)` e `prev_baseline = baseline.predict(X_teste)`;
2. `prev_modelo = busca.predict(X_teste)` (o `GridSearchCV` já reajustou o melhor k em todo o treino).

A função `avaliar` imprime acurácia, precisão, recall e F1.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score, recall_score, f1_score


def avaliar(nome, y_real, y_prev):
    m = (accuracy_score(y_real, y_prev), precision_score(y_real, y_prev, zero_division=0),
         recall_score(y_real, y_prev), f1_score(y_real, y_prev))
    print(f'{nome:<18} acurácia {m[0]:.2f} | precisão {m[1]:.2f} | recall {m[2]:.2f} | F1 {m[3]:.2f}')
    return m


# TODO 5
baseline = None
prev_baseline = None
prev_modelo = None

res_baseline = avaliar('Baseline', y_teste, prev_baseline)
res_modelo = avaliar('k-NN consertado', y_teste, prev_modelo)

## 8 · Antes × depois (pronto)

In [ ]:
print(f'Colega (com os 5 erros): acurácia {melhor_acc:.2f} com k={melhor_k}')
print(f'Consertado:              acurácia {res_modelo[0]:.2f} com k={busca.best_params_["kneighborsclassifier__n_neighbors"]}'
      f' | recall {res_modelo[2]:.2f} | baseline {res_baseline[0]:.2f}')
print(f'\nO resultado "encolheu" {melhor_acc - res_modelo[0]:.2f} de acurácia. O ganho real sobre chutar "baixa" é de {res_modelo[0] - res_baseline[0]:.2f}.')

**Pergunta para discussão:** o resultado consertado é pior. Isso quer dizer que o trabalho piorou?

**Resposta esperada:** não. O número do colega media **outra coisa**: um modelo que vê a resposta (`PCT_COMPARECIMENTO`), avaliado no mesmo teste em que foi escolhido. O número consertado é menor, mas é o que se pode **prometer** para um município novo. Na banca, um resultado modesto e honesto vale mais que um ótimo que cai na primeira pergunta.

## 9 · Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('X só com as 4 colunas de perfil', lambda: list(X.columns) == FEATURES),
    conferir('teste estratificado: 36 municípios, 11 "alta"', lambda: len(y_teste) == 36 and int(y_teste.sum()) == 11),
    conferir('escala dentro do Pipeline', lambda: type(pipeline.steps[0][1]).__name__ == 'StandardScaler'),
    conferir('k escolhido pela CV no treino = 5', lambda: busca.best_params_['kneighborsclassifier__n_neighbors'] == 5),
    conferir('baseline com recall 0 no teste', lambda: res_baseline[2] == 0),
    conferir('o resultado honesto é menor que o do colega', lambda: res_modelo[0] < melhor_acc),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## 10 · Auditoria do projeto do seu grupo (preparação para a banca)
Copie a tabela para uma célula de texto do **notebook do grupo** e preencha com **evidência**: o número da célula, a métrica ou o trecho de código. "Sim" sem evidência não conta.

| Pergunta da banca | Resposta do grupo | Evidência (célula / número) |
|---|---|---|
| 1. Qual é o y? Ele estaria disponível na hora da decisão? | | |
| 2. Qual é o baseline e quanto ele marca? | | |
| 3. Por que essa métrica? Qual erro é mais caro no problema de vocês? | | |
| 4. Onde foi feito o split? O teste foi aberto quantas vezes? | | |
| 5. Há vazamento? (ID, coluna que vem do resultado, preparo fora do Pipeline) | | |
| 6. A conclusão está apoiada em evidência? Qual a oscilação (desvio) na CV? | | |

**Para quem terminar:** rode o experimento do colega consertando **só** o erro 2 (tirar `PCT_COMPARECIMENTO`) e veja quanto a acurácia cai. Qual dos 5 erros "inflava" mais o resultado?